# Xây dựng cây quyết định và rừng cây với Bộ dữ liệu về khả năng vỡ nợ của khách hàng thẻ tín dụng

## 1. Xây dựng cây quyết định bằng thư viện Scikit-Learn

Input: là một DataFrame nơi mỗi hàng là một khách hàng và mỗi cột là một đặc trưng (feature) của khách hàng đó.

Output: 

+ Tên cột: 'default payment next month'

+ Mô tả: Đây là biến mục tiêu (target variable) hay còn gọi là "nhãn" (label). Nó cho mô hình biết khách hàng có bị vỡ nợ (giá trị là 1) hay không (giá trị là 0).

+ Dạng dữ liệu: Nó sẽ là một mảng (hoặc một pandas Series) chỉ chứa các giá trị 0 và 1.

### 1.1 Cài đặt thư viện

In [7]:
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt
#Dòng tiếp theo giúp hiển thị các biểu đồ
%matplotlib inline
import matplotlib as mpl 
mpl.rcParams['figure.dpi'] = 400 
import graphviz 

### 1.2 Đọc dữ liệu

In [8]:
df = pd.read_csv('../datasets/default_of_credit_card_clients.csv') 
features_response = df.columns.tolist()
features_response

['ID',
 'LIMIT_BAL',
 'SEX',
 'EDUCATION',
 'MARRIAGE',
 'AGE',
 'PAY_0',
 'PAY_2',
 'PAY_3',
 'PAY_4',
 'PAY_5',
 'PAY_6',
 'BILL_AMT1',
 'BILL_AMT2',
 'BILL_AMT3',
 'BILL_AMT4',
 'BILL_AMT5',
 'BILL_AMT6',
 'PAY_AMT1',
 'PAY_AMT2',
 'PAY_AMT3',
 'PAY_AMT4',
 'PAY_AMT5',
 'PAY_AMT6',
 'default payment next month']

| Nhóm | Tên cột (Từ ảnh của bạn) | Mô tả chi tiết |
| :--- | :--- | :--- |
| **Biến Mục tiêu** | `'default payment next month'` | **Đây là cột mục tiêu cần dự đoán (Target Variable).** <br> • **1** = Khách hàng có vỡ nợ (default) vào tháng tới. <br> • **0** = Khách hàng không vỡ nợ (thanh toán đủ/đúng hạn). |
| **Thông tin Chung** | `'ID'` | Mã định danh duy nhất cho mỗi khách hàng. |
| **Thông tin Chung** | `'LIMIT_BAL'` | Hạn mức tín dụng được cấp (đơn vị: Đài tệ). |
| **Nhân khẩu học** | `'SEX'` | Giới tính (thường được mã hóa: 1 = Nam, 2 = Nữ). |
| **Nhân khẩu học** | `'EDUCATION'` | Trình độ học vấn (ví dụ: 1 = Cao học, 2 = Đại học, 3 = Trung học, 4 = Khác). |
| **Nhân khẩu học** | `'MARRIAGE'` | Tình trạng hôn nhân (ví dụ: 1 = Đã kết hôn, 2 = Độc thân, 3 = Khác). |
| **Nhân khẩu học** | `'AGE'` | Tuổi của khách hàng (tính bằng năm). |
| **Lịch sử Thanh toán** | `'PAY_0'` | Trạng thái thanh toán của **tháng 9, 2005** (tháng gần nhất). |
| **Lịch sử Thanh toán** | `'PAY_2'` | Trạng thái thanh toán của **tháng 8, 2005**. |
| **Lịch sử Thanh toán** | `'PAY_3'` | Trạng thái thanh toán của **tháng 7, 2005**. |
| **Lịch sử Thanh toán** | `'PAY_4'` | Trạng thái thanh toán của **tháng 6, 2005**. |
| **Lịch sử Thanh toán** | `'PAY_5'` | Trạng thái thanh toán của **tháng 5, 2005**. |
| **Lịch sử Thanh toán** | `'PAY_6'` | Trạng thái thanh toán của **tháng 4, 2005** (tháng xa nhất). |
| **Lịch sử Sao kê** | `'BILL_AMT1'` | Số tiền trên sao kê của **tháng 9, 2005** (tháng gần nhất). |
| **Lịch sử Sao kê** | `'BILL_AMT2'` | Số tiền trên sao kê của **tháng 8, 2005**. |
| **Lịch sử Sao kê** | `'BILL_AMT3'` | Số tiền trên sao kê của **tháng 7, 2005**. |
| **Lịch sử Sao kê** | `'BILL_AMT4'` | Số tiền trên sao kê của **tháng 6, 2005**. |
| **Lịch sử Sao kê** | `'BILL_AMT5'` | Số tiền trên sao kê của **tháng 5, 2005**. |
| **Lịch sử Sao kê** | `'BILL_AMT6'` | Số tiền trên sao kê của **tháng 4, 2005** (tháng xa nhất). |
| **Lịch sử Chi trả** | `'PAY_AMT1'` | Số tiền khách hàng đã thanh toán vào **tháng 9, 2005** (tháng gần nhất). |
| **Lịch sử Chi trả** | `'PAY_AMT2'` | Số tiền khách hàng đã thanh toán vào **thàg 8, 2005**. |
| **Lịch sử Chi trả** | `'PAY_AMT3'` | Số tiền khách hàng đã thanh toán vào **tháng 7, 2005**. |
| **Lịch sử Chi trả** | `'PAY_AMT4'` | Số tiền khách hàng đã thanh toán vào **tháng 6, 2005**. |
| **Lịch sử Chi trả** | `'PAY_AMT5'` | Số tiền khách hàng đã thanh toán vào **tháng 5, 2005**. |
| **Lịch sử Chi trả** | `'PAY_AMT6'` | Số tiền khách hàng đã thanh toán vào **tháng 4, 2005** (tháng xa nhất). |

#### Lưu ý quan trọng khi phân tích
Dòng thời gian (Timeline): Đây là điều quan trọng nhất. Các cột có số 1 (hoặc 0 trong PAY_0) là tháng gần nhất (tháng 9/2005). Các cột có số 6 là tháng xa nhất (tháng 4/2005). Dữ liệu đi lùi về quá khứ.

Ý nghĩa của các cột PAY_...: Đây là các cột cực kỳ quan trọng và có ý nghĩa đặc biệt:

+ -2: Không có giao dịch / không sử dụng thẻ trong tháng.

+ -1: Đã thanh toán đầy đủ (Paid in full).

+ 0: Đã thanh toán số tiền tối thiểu (Minimum payment) hoặc sử dụng hết hạn mức (Credit limit).

+ 1: Trễ hạn thanh toán 1 tháng.

+ 2: Trễ hạn thanh toán 2 tháng.

... (và tương tự cho đến 8 hoặc 9 tháng).

### 1.3 Tiền xử lý dữ liệu

#### Lựa chọn đặc trưng

In [9]:
#Lập danh sách các cột cần xóa không phải là tính năng hoặc biến phản hồi
items_to_remove = ['ID', 'SEX', 'PAY_2', 'PAY_3',\
'PAY_4', 'PAY_5', 'PAY_6',\
'EDUCATION_CAT', 'graduate school',\
'high school', 'none',\
'others', 'university']

features_response = [item for item in features_response if item not
in items_to_remove]
features_response

['LIMIT_BAL',
 'EDUCATION',
 'MARRIAGE',
 'AGE',
 'PAY_0',
 'BILL_AMT1',
 'BILL_AMT2',
 'BILL_AMT3',
 'BILL_AMT4',
 'BILL_AMT5',
 'BILL_AMT6',
 'PAY_AMT1',
 'PAY_AMT2',
 'PAY_AMT3',
 'PAY_AMT4',
 'PAY_AMT5',
 'PAY_AMT6',
 'default payment next month']

Bộ đặc trưng mới tập trung vào một giả thuyết:

"Khả năng vỡ nợ chủ yếu được quyết định bởi tình trạng nhân khẩu học cơ bản (Tuổi, Học vấn, Hôn nhân), hạn mức tín dụng (LIMIT_BAL), toàn bộ lịch sử về SỐ TIỀN (BILL_AMT và PAY_AMT trong 6 tháng), nhưng chỉ cần TRẠNG THÁI thanh toán của tháng gần nhất (PAY_0)."

### 1.4 Chuẩn bị dữ liệu cho tập train và tập test

In [10]:
from sklearn.model_selection import train_test_split
from sklearn import tree
#Chia dữ liệu thành các tập huấn luyện và thử nghiệm bằng cách sử dụng cùng một hạt giống ngẫu nhiên

X_train, X_test, y_train, y_test = \
train_test_split(df[features_response[:-1]].values,
df['default payment next month'].values,
test_size=0.2, random_state=24)


Bước này chia bộ dữ liệu thành 2 phần một cách ngẫu nhiên:

+ Tập Huấn luyện (Training Set): Phần lớn dữ liệu (80%) dùng để dạy hoặc "fit" mô hình (**test_size=0.2**).

+ Tập Kiểm tra (Test Set): Phần nhỏ dữ liệu (20%) được giữ bí mật với mô hình. Nó chỉ được dùng một lần duy nhất ở cuối cùng để đánh giá xem mô hình hoạt động tốt đến đâu trên dữ liệu mới mà nó chưa từng thấy

### 1.5 Xây dựng cây quyết định từ lớp DecisionTreeClassifier có trong thư viện Scikit-Learn

In [11]:
# Cây sẽ phát triển đến độ sâu tối đa là 2
dt = tree.DecisionTreeClassifier(max_depth=2)
dt.fit(X_train, y_train)

DecisionTreeClassifier(max_depth=2)

Khởi tạo Mô hình với max_depth=2:

+ Chống Overfitting (Học vẹt): Đây là lý do quan trọng nhất. Chúng ta hy sinh một chút độ chính xác trên tập train để đổi lấy độ chính xác cao hơn (và đáng tin cậy hơn) trên tập test.

+ Tăng Tính Diễn Giải (Interpretability): Mô hình không phải là "hộp đen". Bạn có thể vẽ nó ra, và bất kỳ ai (kể cả sếp của bạn) cũng có thể hiểu 100% logic mà nó dùng để ra quyết định. Điều này cực kỳ quan trọng trong các ngành như tài chính, y tế.

+ Tăng Tốc độ (Speed): Một cái cây nông huấn luyện và dự đoán nhanh hơn rất nhiều, tốn ít tài nguyên tính toán hơn.

Kết luận: chọn max_depth=2 không phải vì nó cho độ chính xác cao nhất trên lý thuyết, mà vì nó tạo ra một mô hình hữu ích, an toàn, dễ hiểu và có khả năng áp dụng vào thực tế tốt.

### 1.6 Hiển thị cây quyết định với package graphviz

In [15]:
dot_data = tree.export_graphviz(dt, 
                                out_file=None,
                                filled=True,
                                rounded=True,
                                feature_names=\
                                features_response[:-1],
                                proportion=True,
                                class_names=['Not defaulted', 'Defaulted'])
graph = graphviz.Source(dot_data)
print("Kết quả thu được: ")
graph

Kết quả thu được: 


ExecutableNotFound: failed to execute WindowsPath('dot'), make sure the Graphviz executables are on your systems' PATH